In [ ]:
import json
import pandas as pd
wrong_drs=pd.read_csv("cmip6_list_data_ref_syntax_wrong-drs-by-pids.txt")

In [ ]:
drs_keys=['activity_id', 'institution_id', 'source_id', 'experiment_id', 'member_id', 'table_id', 'variable_id', 'grid_label', 'version']
drs_keys_noversion=['activity_id', 'institution_id', 'source_id', 'experiment_id', 'member_id', 'table_id', 'variable_id', 'grid_label']

In [ ]:
wrong_drs=pd.DataFrame([k.split('.') for k in wrong_drs["data_ref_syntax"]], columns=drs_keys)

In [ ]:
sort_out=wrong_drs[(wrong_drs["variable_id"].str.contains("\?")) |
          (wrong_drs["variable_id"]=='')]
sort_out.agg('.'.join,axis=1).to_csv("cmip6_list_data_ref_syntax_sortout-varwrong.txt", index=False)

In [ ]:
wrong_drs=wrong_drs[~wrong_drs.index.isin(sort_out.index)]

In [ ]:
duplicates=wrong_drs[wrong_drs[drs_keys_noversion].duplicated(keep=False)]
len(duplicates)
duplicates[drs_keys]

In [ ]:
duplicates[drs_keys].agg('.'.join,axis=1).to_csv(
    "cmip6_list_data_ref_syntax_sortout-versionduplicated.txt", index=False)

In [ ]:
#wrong_drs=wrong_drs[~wrong_drs[drs_keys_noversion].duplicated()]

In [ ]:
wrong_drs

In [ ]:
import intake
cmip6_col = intake.open_esm_datastore("https://gitlab.dkrz.de/data-infrastructure-services/intake-esm/-/raw/master/esm-collections/cloud-access/dkrz_cmip6_disk.json")
cmip6_col.df.head()

In [ ]:
cmip6_cat=cmip6_col.df[drs_keys]
#df is the dataframe under the catalog
#each dataset consists of many files. by keeping only one we get the list of datasets:
cmip6_cat=cmip6_col.df[drs_keys].drop_duplicates(keep="first").reindex()

In [ ]:
wrong_drs_idx=wrong_drs.set_index(drs_keys_noversion).index
cmip6_cat_idx=cmip6_cat.set_index(drs_keys_noversion).index

In [ ]:
cat_candidates_version=cmip6_cat[cmip6_cat_idx.isin(wrong_drs_idx)]

In [ ]:
cat_candidates_version

In [ ]:
cat_candidates_version_idx=cat_candidates_version.set_index(drs_keys_noversion).index

In [ ]:
wrong_drs_candidates_version=wrong_drs[wrong_drs_idx.isin(cat_candidates_version_idx)].reset_index(drop=True)

In [ ]:
wrong_drs_candidates_version_idx=wrong_drs_candidates_version.set_index(drs_keys_noversion).index

In [ ]:
wrong_drs_candidates_version.loc[:,"data_ref_syntax"]=wrong_drs_candidates_version.agg('.'.join,axis=1)
cat_candidates_version.loc[:,"data_ref_syntax"]=cat_candidates_version[drs_keys].agg('.'.join,axis=1)

In [ ]:
for row,idx in enumerate(wrong_drs_candidates_version_idx) :
    wrong_drs_candidates_version.loc[row,"candidate"]=','.join(cat_candidates_version.set_index(drs_keys_noversion).loc[idx,"data_ref_syntax"].values)

In [ ]:
wrong_drs_candidates_version_dict={}
for i,row in wrong_drs_candidates_version[["data_ref_syntax","candidate"]].iterrows():
    wrong_drs_candidates_version_dict[row["data_ref_syntax"]]=row["candidate"]

In [ ]:
with open("cmip6_list_data_ref_syntax_drs-candidates-by-version.json", 'w') as f:
    json.dump(wrong_drs_candidates_version_dict,f,indent=4, sort_keys=True)

In [ ]:
wrong_drs=wrong_drs[~wrong_drs_idx.isin(wrong_drs_candidates_version_idx)]
wrong_drs.agg('.'.join,axis=1).to_csv("cmip6_list_data_ref_syntax_sortout-notfound.txt", index=False)